# Crowd Density Estimation – MG Road Video

This notebook presents a complete workflow for estimating crowd density using YOLOv8 on a self-captured video (`Nandi.mp4`). The pipeline includes frame extraction, annotation, model training, inference, and analysis of crowd density using visual graphs.

**Tools & Libraries Used:** YOLOv8 (Ultralytics), OpenCV, Matplotlib, Python.

**Objective:** Detect and count people in a real-world video and classify the crowd density per frame as:
- **Low** (0–10 people)
- **Medium** (11–25 people)
- **High** (26+ people)


### Import Libraries
We import all the necessary libraries required for model loading, prediction, image processing, plotting, and file handling.

In [ ]:
from ultralytics import YOLO
import cv2
import os
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display


### 🎞️ Frame Extraction from Video

To prepare the dataset for annotation, frames were extracted from the `Nandi.mp4` video using OpenCV. These frames were manually annotated with bounding boxes using [LabelImg/Roboflow].

Below is the code used to extract and save frames at 1 frame per second.

In [ ]:
import cv2
import os

videos = ['MG Road.mp4']  
output_dir = 'frames'
os.makedirs(output_dir, exist_ok=True)

frame_rate = 5  

for video in videos:
    vidcap = cv2.VideoCapture(video)
    success, image = vidcap.read()
    count = 0
    saved = 0
    while success:
        if count % int(vidcap.get(cv2.CAP_PROP_FPS)) // frame_rate == 0:
            cv2.imwrite(f"{output_dir}/{video}_{saved}.jpg", image)
            saved += 1
        success, image = vidcap.read()
        count += 1
    vidcap.release()


### 🖊️ Image Annotation Summary

A total of 50+ frames were manually annotated using CVAT. Each person in every frame was labeled using bounding boxes. The dataset was exported in **YOLO format** with separate folders for `images` and `labels`.

Annotations were manually corrected to ensure high-quality ground truth.

### 🧠 Model Training

The YOLOv8 model was trained on the annotated dataset using transfer learning from pretrained COCO weights. Training was done with 50 epochs and early stopping.

The training was performed using the Ultralytics CLI:

```bash
yolo detect train data=config.yaml model=yolov8n.pt epochs=100 imgsz=640

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  


model.train(
    data="data.yaml",          
    epochs=50,                 
    imgsz=640,                 
    batch=8,                   
    project="crowd_detection_yolov8",  
    name="mg_road_model",     
    verbose=True              
)


### Loading Trained Model and Predict on Video

We now load our YOLOv8 trained model and run predictions on the MG Road video.

In [ ]:
from ultralytics import YOLO

model = YOLO("crowd_detection_yolov8/mg_road_model3/weights/best.pt")

model.predict(
    source="MG Road.mp4",
    save=True,
    save_txt=True,   
    conf=0.4,
    show=True
)


### 🎥 Output Video – MG Road Crowd Detection

The output video shows bounding boxes for detected people in each frame. Due to limitations in VS Code, inline playback is not supported. Please open the output file manually from:

📁 `runs/detect/predict2/MG Road Output.mp4`

You can also play it by clicking or running the code below:


In [ ]:
import webbrowser
webbrowser.open("runs/detect/predict2/MG Road Output.mp4")

### 📈 Crowd Density Estimation – Processing & Analysis

Each frame's detection data was analyzed to:

- Count the number of people per frame.
- Classify the crowd density level:
  - Low: 0–10
  - Medium: 11–25
  - High: 26+
- Save the data to a `.csv` file for analysis.
- Generate time-series and pie charts to visualize crowd trends.

In [ ]:
from IPython.display import Image, display

display(Image(filename="count_per_frame.png"))
display(Image(filename="density_distribution.png"))


### 📋 Conclusion

- The YOLOv8 model successfully detected and counted people in the MG Road video.
- The People Count per Frame graph shows that most frames had between 6 to 11 people, with occasional drops to 4–5 people. This indicates consistent but moderate pedestrian traffic.
- The Crowd Density Distribution pie chart confirms that the vast majority of frames fall under the Low density category (0–10 people), accounting for 97.2% of all frames.
- A small portion (2.8%) was classified as Medium density (11–25 people), suggesting short bursts of slightly heavier foot traffic.
- The project demonstrates how deep learning and video analytics can be used for smart city monitoring.
- Overall, the crowd activity at MG Road during the video capture was relatively sparse, and the model performed accurately in detecting and classifying individuals under varying conditions.